**Overall Purpose**

The code fine-tunes a pre-trained Geneformer model to classify pancreatic cell types from single-cell RNA-seq data that has been tokenized into gene sequences.

Key Sections:

**1. Setup & Environment**

Installs dependencies (scanpy, datasets, etc.)

Clones Geneformer repository

Sets up CUDA memory configuration

Mounts Google Drive for data access

**2. Data Loading & Exploration**

Loads tokenized pancreatic dataset

```
pancreas_dataset = "/content/drive/My Drive/geneformer_test2/pancreas_scib.dataset"
token_dataset = load_from_disk(pancreas_dataset)
```

Data contains input_ids (tokenized genes), celltype, and other metadata

Explores cell type distribution with visualizations

**3. Data Preprocessing Pipeline**

Filter rare cell types (>10% threshold)

```
clusters_to_keep = [k for k, v in celltype_counter.items() if v > min_cells]

# Keep only full-length sequences (2048 tokens)
filtered_data = filtered_data.filter(lambda x: x["length"] == 2048, num_proc=16)

# Downsample to 1000 cells to avoid memory issues
target_total = 1000
```

**4. Memory Management**

Aggressive downsampling (1000 cells) to prevent CUDA OOM

Uses proportional sampling to maintain cell type ratios

Implements gradient checkpointing and other memory optimizations

**5. Label Preparation**

Create numerical labels for training

target_names = sorted(list(set(train_data["label"])))

celltype_label_dict = {name: idx for idx, name in enumerate(target_names)}

**6. Model Training**

Load pre-trained Geneformer

```
model = BertForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=len(celltype_label_dict.keys()),
    output_attentions=False,
    output_hidden_states=False
).to("cuda")

```

**7. Training Configuration**

Uses conservative settings to avoid memory issues

Implements gradient accumulation and checkpointing

Includes comprehensive evaluation metrics

**Key Features:**

*Memory Optimization:*

Dataset downsampling (1000 cells)

Gradient checkpointing enabled

Batch size = 5 with gradient accumulation

Sequence length capped at 2048

*Data Quality:*

Filters rare cell types (>10% threshold)

Only uses full-length sequences

Maintains proportional cell type distribution

*Biological Relevance:*

Focuses on pancreatic islet cells (beta, alpha, delta, etc.)

Includes cell type distribution analysis

Provides biological context for each cell type

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!pip install numpy==1.26.4
import os

In [ ]:
!pip install scanpy
!pip install loompy
!pip install datasets

In [ ]:
!git lfs install
!git clone https://huggingface.co/ctheodoris/Geneformer
!ls -1

In [ ]:
%cd Geneformer
!pip install -e .

In [ ]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

**Load and check the input .dataset after tokenization**

In [ ]:
# Copy from Google Drive

import loompy
import scanpy
import datasets
from datasets import load_from_disk

pancreas_dataset = "/content/drive/My Drive/geneformer_test2/pancreas_scib_V2model.dataset"
token_dataset = load_from_disk(pancreas_dataset)

# Print basic info
print("Type of loaded dataset:", type(token_dataset))
print("Dataset summary:")
print(token_dataset)

# Print the feature schema (columns and types)
print("Dataset features:")
print(token_dataset.features)

# Preview the first 2 records
print("\nFirst 2 records:")
print(token_dataset[:2])

# Convert to pandas DataFrame and display the first few rows
df = token_dataset.to_pandas()
print("\nHead of the dataset as pandas DataFrame:")
print(df.head())

# input_ids
# A list of integers, where each integer represents a tokenized gene (mapped from an Ensembl ID) using Geneformer's tokenizer.
# These are the input tokens for the transformer model (similar to word tokens in NLP).

original_dataset = token_dataset

In [ ]:
!ls -1 geneformer

**input_ids** - tokenized expression (int IDs representing Ensembl genes)

**cell metadata** - celltype, tech, n_counts, obs_names, batch, etc.

In a Geneformer .dataset object (loaded with load_from_disk()), you cannot directly see gene names — only the input_ids, which are integers (tokens) that represent genes.

These token IDs were created during tokenization by mapping Ensembl gene IDs to integers using a token dictionary. The gene names themselves are not stored inside the .dataset — but - you can recover gene names in the following way :

```
import pickle

with open("/content/Geneformer/geneformer/token_dictionary_gc104M.pkl", "rb") as f:
    token_dict = pickle.load(f)

# Reverse it: token ID → gene (usually Ensembl ID)
token_to_gene = {v: k for k, v in token_dict.items()}

```

In [ ]:
# Save the dataset to the new folder
token_dataset.save_to_disk("/content/pancreas_scib.dataset")
print(f"Dataset successfully saved to /content/pancreas_scib.dataset")

In [ ]:
print(df["input_ids"].head())
print(df["input_ids"].tail())

In [ ]:
import os
import subprocess
os.environ["NCCL_DEBUG"] = "INFO"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = ""

In [ ]:
def find_gpus(nums=1):
    if nums == 1:
        return ['0']

    os.system('nvidia-smi -q -d Memory | grep Free >tmp_free_gpus')
    with open('tmp_free_gpus', 'r') as lines_txt:
        frees = lines_txt.readlines()
        idx_freeMemory_pair = [ (idx,int(x.split()[2]))
                              for idx,x in enumerate(frees) ]

    idx_freeMemory_pair_correct = []
    step = int(len(idx_freeMemory_pair) / nums)
    for i in range(0, len(idx_freeMemory_pair), step):
        obj = idx_freeMemory_pair[i]
        obj = (int(obj[0]/step), obj[1])
        idx_freeMemory_pair_correct.append(obj)

    idx_freeMemory_pair_correct.sort(key=lambda my_tuple:my_tuple[1],reverse=True)
    usingGPUs = [str(idx_memory_pair[0])
                    for idx_memory_pair in idx_freeMemory_pair_correct[:nums] ]

    print('using GPU idx: #', usingGPUs)
    return usingGPUs

n_gpus = str(subprocess.check_output(["nvidia-smi", "-L"])).count('UUID')
os.environ['CUDA_VISIBLE_DEVICES'] = ','.join(find_gpus(nums=n_gpus)[0])

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import datetime
import glob
import pickle
import subprocess

import numpy as np
import pandas as pd
import seaborn as sns

from collections import Counter, defaultdict

from datasets import Dataset, DatasetDict, load_from_disk

from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

from transformers import BertForSequenceClassification, Trainer
from transformers.training_args import TrainingArguments

from geneformer import Classifier, DataCollatorForCellClassification, EmbExtractor

***Set up environment variables***

In [ ]:
import sys, os
LIB_PATH=sys.path[1].split("/lib/")[0]
LFS_HOME=f"{LIB_PATH}/bin/git-lfs"
print(LFS_HOME)
# os.system(f"{LFS_HOME} install")
# os.system(f"{LFS_HOME} clone https://huggingface.co/ctheodoris/Geneformer")
# os.system(f"cd ./Geneformer && {LFS_HOME} pull")

In [ ]:
!echo -e "\nThe files in the folder : pancreas_scib.dataset\n"
!ls -1 /content/pancreas_scib.dataset

In [ ]:
!ls -1 ./geneformer/

In [ ]:
# Pre-train model directory

# pre_model_dir = glob.glob(os.path.join(repo_dir,"*-12L-30M*"))[0]   ### use 12-layer Geneformer model
# pre_model_dir = glob.glob(os.path.join(repo_dir, "*-6L-30M*"))[0]  ### use 6-layer Geneformer model

# Input data set path
# input_prefix = "pancreas_scib"

# gene_name_id_dict_gc104M.pkl

# ensembl_mapping_dict_gc104M.pkl
# gene_median_dictionary_gc104M.pkl
# token_dictionary_gc104M.pkl

# pancreas_dataset = token_dataset
original_dataset = token_dataset

# Specific paths

token_dictionary = "/content/Geneformer/geneformer/token_dictionary_gc104M.pkl"
model_v1 = "/content/Geneformer/Geneformer-V1-10M/"
model_v2 = "/content/Geneformer/Geneformer-V2-104M"

input_dir = "/content/pancreas_scib.dataset"
output_dir = "/content/pancreas_scib.classification_output"
input_prefix = "pancreas_scib"

In [ ]:
print("Loading pancreas dataset...")

# Load the main pancreas dataset
original_dataset = load_from_disk(f"/content/pancreas_scib.dataset/")

print(f"Total cells loaded: {len(original_dataset)}")
print(f"Dataset features: {original_dataset.features}")

# Check if celltype column exists
if "celltype" in original_dataset.features:
    print("✅ 'celltype' is present in the dataset.")
else:
    print("❌ 'celltype' is NOT present in the dataset.")
    print("Available columns:", list(original_dataset.features.keys()))

In [ ]:
# The Counter is used here to count the frequency of each cell type in your dataset. Let me explain why this is important:

import matplotlib.pyplot as plt

celltype_counter = Counter(original_dataset["celltype"])
print(f"\n📊 Cell type distribution (raw counts): {celltype_counter}")

plt.figure(figsize=(6, 4))
plt.bar(celltype_counter.keys(), celltype_counter.values())
plt.xlabel("Cell Type")
plt.ylabel("Count")
plt.title("Cell Type Distribution")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# 🔢 Calculate and print percentages
total = sum(celltype_counter.values())
percentages = {cell: f"{100 * count / total:.1f}%" for cell, count in celltype_counter.items()}
print("\n📈 Cell type percentages:")
for cell_type, pct in percentages.items():
    print(f"{cell_type}: {pct}")

In [ ]:
original_dataset = original_dataset.map(lambda x: {"label": x["celltype"]})
print(original_dataset.shape)

output_prefix = "pancreas_labeled"

# Save dataset
original_dataset.save_to_disk(f"/content/{output_prefix}.dataset")

print(f"✅ Saved shuffled dataset to: /content/{output_prefix}.dataset")

print(f"📊 Number of cells in original_dataset: {len(original_dataset)}")

for i in range(min(2, len(original_dataset))):
    print(f"\n📄 Cell #{i}:")
    print(original_dataset[i])


In [ ]:
# Keep cell types that appear in more than 0.1 of training data

threshold = 0.1  # 10% expressed as a decimal
min_cells = len(original_dataset) * threshold

clusters_to_keep = [k for k, v in celltype_counter.items() if v > min_cells]
print(f"\n✅ Cell types to keep (>0.1% threshold = {int(min_cells)} cells): {len(clusters_to_keep)} types")
print(f"🧪 Retained cell types: {clusters_to_keep}")

# ℹ️ Quick breakdown of pancreatic islet cell types
print("""
🔬 Pancreatic islet cell functions:
- Beta (β): secrete insulin (lowers blood glucose)
- Alpha (α): secrete glucagon (raises blood glucose)
- Delta (δ): secrete somatostatin (inhibits insulin and glucagon)
- PP (γ/F): secrete pancreatic polypeptide
- Epsilon (ε): secrete ghrelin
""")

# 📊 Compute and display filtered cell type distribution
total_cells = sum(celltype_counter[k] for k in clusters_to_keep)

print("📈 Cell type distribution (filtered):")
for k in clusters_to_keep:
    count = celltype_counter[k]
    pct = 100 * count / total_cells
    print(f"{k}: {count} cells ({pct:.2f}%)")


**To Identify Rare Cell Types**

clusters_to_keep = [k for k, v in celltype_counter.items() if v > len(train_data) * 0.005]

and to filter out cell types that appear in less than 0.5% of your data, and to prevent training problems.

Class imbalance: If you have 10,000 acinar cells but only 5 rare cells, the model will be biased.

Insufficient samples: Can't learn meaningful patterns from very few examples

Poor generalization: Model might overfit to rare classes.

In [ ]:
print(f"📦 Original dataset size: {len(original_dataset)}")

In [ ]:
# Step 1: Filter data to keep only frequent cell types
filtered_data = original_dataset.filter(lambda x: x["celltype"] in clusters_to_keep, num_proc=16)
print(filtered_data.shape)

# Step 2: Add a new column 'label' that copies 'celltype'
filtered_data = filtered_data.map(lambda x: {"label": x["celltype"]})
print(filtered_data.shape)

# Step 3: Shuffle the dataset for unbiased splits
shuffled_data = filtered_data.shuffle(seed=42)
print(shuffled_data.shape)

# Step 4: Inspect the DataFrame
df = shuffled_data.to_pandas()
print(df.head())
print(f"\n📊 Shape of shuffled dataset: {df.shape}")

print(f"✅ Number of cells in shuffled_data: {len(shuffled_data)}")
for i in range(min(3, len(shuffled_data))):  # Change 10 to however many you want
    print(f"\n📄 Cell #{i}:")
    print(shuffled_data[i])

# ✅ Now shuffled_data will contain both celltype and label columns.

# instead of Renaming (shuffle + rename_column) :
# shuffled_data = filtered_data.shuffle(seed=42).rename_column("celltype", "label")

# Why rename to "label"?
# Standard ML convention - most training frameworks expect a column called "label"
# HuggingFace compatibility - Trainer expects numerical labels in a "label" column
# Prepares for next step - where we'll convert text labels to numbers

# Why shuffle?
# Randomizes order of cells in the dataset
# Prevents bias during train/eval splits
# seed=42 ensures reproducible results

# Why do we shuffle a dataset?

# Avoid Ordering Bias
# Biological or technical datasets may have samples grouped by condition, timepoint, or batch.
# If we don't shuffle, training data might be biased (e.g., early batches all beta cells, later batches all alpha cells).

# Improve Generalization
# Helps models learn robust patterns by seeing more diverse examples early and throughout training.

# Ensure Balanced Batches
# Training in mini-batches benefits from a mix of classes or cell types.
# Shuffling helps maintain class balance across batches.

# Prevents Overfitting
# Learning from patterns in ordering (like repeated samples from the same donor) may cause overfitting.
# Shuffling breaks those patterns.

In [ ]:
# To save your shuffled_data dataset to disk after filtering, labeling, and shuffling, simply use the save_to_disk() method provided by Hugging Face datasets.

# Define output path
output_prefix = "pancreas_shuffled_filtered_labeled"

# Save dataset
shuffled_data.save_to_disk(f"/content/{output_prefix}.dataset")

print(f"✅ Saved shuffled dataset to: /content/{output_prefix}.dataset")

In [ ]:
!ls -1 /content/pancreas_shuffled_filtered_labeled.dataset/
# Copy the folder
!cp -r /content/pancreas_shuffled_filtered_labeled.dataset/ /content/drive/MyDrive/geneformer_test2

To downsize the data to 3000 cells, keeping the same % of cells, nin order not to get CUDA out of memory ("CUDA out of memory. Tried to allocate 1.56 GiB. GPU 0 has a total capacity of 39.56 GiB of which 404.88 MiB is free. Process 5476 has 39.15 GiB memory in use. Of the allocated memory 37.55 GiB is allocated by PyTorch, and 1.11 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management")  

In [ ]:
# The percentage of each cell type in the shuffled_data

from collections import Counter

# Total target cells
target_total = 2000

# Count cell types
celltype_counter = Counter(shuffled_data["celltype"])

# Compute percentages
total_cells = sum(celltype_counter.values())
percentages = {cell: 100 * count / total_cells for cell, count in celltype_counter.items()}

# Print results
print(f"\n📊 Cell type distribution in shuffled dataset (total = {total_cells} cells):")
for cell_type, count in celltype_counter.items():
    pct = percentages[cell_type]
    print(f"{cell_type}: {count} cells ({pct:.2f}%)")

import matplotlib.pyplot as plt

# === Pie Chart Visualization ===
labels = [f"{cell} ({count}, {percentages[cell]:.1f}%)" for cell, count in celltype_counter.items()]
sizes = list(celltype_counter.values())

plt.figure(figsize=(5, 5))
plt.pie(sizes, labels=labels, startangle=140, wedgeprops={"edgecolor": "white"})
plt.title(f"📊 Cell Type Distribution (n = {total_cells} cells)", fontsize=10)
plt.axis("equal")  # Equal aspect ratio ensures the pie is round
plt.tight_layout()
plt.show()


**Downsampling**

In [ ]:
# ✅ Next step: Create the downsampled dataset
from datasets import concatenate_datasets

# === Compute sample sizes based on existing proportions ===
sample_sizes = {
    k: int(round(v / total_cells * target_total)) for k, v in celltype_counter.items()
}

# Adjust to ensure total is exactly target_total (e.g., 2000)
diff = target_total - sum(sample_sizes.values())
if diff != 0:
    largest = max(sample_sizes, key=sample_sizes.get)
    sample_sizes[largest] += diff

# === Downsample each group ===
downsampled_groups = [
    shuffled_data.filter(lambda x: x["celltype"] == k).shuffle(seed=42).select(range(sample_sizes[k]))
    for k in sample_sizes
]

# === Combine downsampled groups ===
downsampled_dataset = concatenate_datasets(downsampled_groups)

# === Check final cell distribution ===
final_counts = Counter(downsampled_dataset["celltype"])
print(f"\n✅ Final downsampled dataset (n = {target_total}):")
for k, v in final_counts.items():
    print(f"{k}: {v} cells ({100 * v / target_total:.2f}%)")

print(f"🎯 Downsampled dataset contains {len(downsampled_dataset)} cells.")
print("📊 Distribution:")
from collections import Counter
from pprint import pprint

ct_counts = Counter(downsampled_dataset["celltype"])
pprint({k: f"{v} cells ({v / len(downsampled_dataset) * 100:.2f}%)" for k, v in ct_counts.items()})

In [ ]:
import matplotlib.pyplot as plt

# 🎯 final_counts is already computed using:
# final_counts = Counter(downsampled_dataset["celltype"])

# ✅ Prepare data
labels = list(final_counts.keys())
sizes = list(final_counts.values())

# 🥧 Plot pie chart
plt.figure(figsize=(4, 4))
plt.pie(
    sizes,
    labels=labels,
    autopct='%1.1f%%',
    startangle=140
)
plt.title("📊 Cell Type Distribution in Downsampled Dataset")
plt.axis('equal')  # Make it a perfect circle
plt.tight_layout()
plt.show()

# Show full content of the first two cells
print("\n🧬 Cell #0:")
print(downsampled_dataset[0])

print("\n🧬 Cell #1:")
print(downsampled_dataset[1])


**Downsampled_dataset**

In [ ]:
for i in range(2):
    cell = downsampled_dataset[i]
    print(f"\n🧬 Cell #{i}:")
    print(f"obs_names : {cell['obs_names']}")
    print(f"celltype  : {cell['celltype']}")
    print(f"length    : {cell['length']}")
    print(f"input_ids : {cell['input_ids'][:5]}...")


In [ ]:
# To save your shuffled_data dataset to disk after filtering, labeling, and shuffling, simply use the save_to_disk() method provided by Hugging Face datasets.

# Define output path
output_prefix = "pancreas_shuffled_filtered_labeled_downsampled"

# Save dataset
downsampled_dataset.save_to_disk(f"/content/{output_prefix}.dataset")

print(f"✅ Saved shuffled dataset to: /content/{output_prefix}.dataset")

In [ ]:
!ls -1 /content/pancreas_shuffled_filtered_labeled_downsampled.dataset
# Copy the folder
!cp -r /content/pancreas_shuffled_filtered_labeled_downsampled.dataset/ /content/drive/MyDrive/geneformer_test2

In [ ]:
# ============================================================================
# DATA PREP
# ============================================================================

# Load and prepare dataset (pancreas dataset)
# original_dataset = load_from_disk(f"{input_dir}/pancreas_dataset.dataset")

# The train_test_split method from the 🤗 Hugging Face datasets library,
# which operates on a Dataset object (like original_dataset).

# x["train"] → 80%
# x["test"] → 20%

# An example :

# x = original_dataset.train_test_split(test_size=0.2, seed=42)
# train_data = x["train"]
# test_data = x["test"]

# Print summary
# print("Training set:")
# print(train_data)
# print("Number of samples:", len(train_data))

# print("\nTest set:")
# print(test_data)
# print("Number of samples:", len(test_data))

# from pprint import pprint

# Print the first 4 rows of the training set
# pprint(train_data[:4])

# Print the first 4 rows of the testing set
# pprint(test_data[:4])

# for i in range(4):
#    print({k: train_data[i][k] for k in ["celltype", "n_counts", "length"]})

# for i in range(4):
#    print({k: test_data[i][k] for k in ["celltype", "n_counts", "length"]})

# The length refers to the number of tokens (gene IDs) used to represent each cell after tokenization.
# This is what Geneformer uses as the input sequence length for each cell. It is capped (often at 2048) to fit the model's maximum input size.

In [ ]:
# Now, downsampled_dataset is the actual object to use going forward — for example, for splitting into train/test:

# x = downsampled_dataset.train_test_split(test_size=0.2, seed=42)
# pancreas_trainset = x["train"]
# pancreas_evalset = x["test"]

train_data = downsampled_dataset

**Recommended: Stratified Split by Cell Type**

To ensure the same percentage of each cell type (i.e., maintain class balance) between training and evaluation sets, you should do **a stratified split** — not just a range-based split like: `pancreas_trainset = downsampled_dataset.select(range(train_size))`
That approach doesn’t preserve cell type proportions.

https://geneformer.readthedocs.io/en/latest/geneformer.classifier.html

**prepare_data**

Prepare data for cell state or gene classification.

Parameters

**input_data_filePath**

Path to directory containing .dataset input

**output_directoryPath**

Path to directory where prepared data will be saved

**output_prefixstr**

Prefix for output file

**split_id_dict : None, dict**

Dictionary of IDs for train and test splits

Three-item dictionary with keys: attr_key, train, test

**attr_key:** key specifying name of column in .dataset that contains the IDs for the data splits

**train:** list of IDs in the attr_key column to include in the train split

**test:** list of IDs in the attr_key column to include in the test split

For example:

{“attr_key”: “individual”, “train”: [“patient1”, “patient2”, “patient3”, “patient4”], “test”: [“patient5”, “patient6”]}

**test_size : None, float**

Proportion of data to be saved separately and held out for test set
(e.g. 0.2 if intending hold out 20%)

If None, will inherit from split_sizes[“test”] from Classifier

The training set will be further split to train / validation in self.validate
Note: only available for CellClassifiers

**attr_to_split:** None, str

Key for attribute on which to split data while balancing potential confounders
e.g. “patient_id” for splitting by patient while balancing other characteristics
Note: only available for CellClassifiers

**attr_to_balance:** None, list

List of attribute keys on which to balance data while splitting on attr_to_split
e.g. [“age”, “sex”] for balancing these characteristics while splitting by patient

Note: only available for CellClassifiers

**max_trials** : None, int

Maximum number of trials of random splitting to try to achieve balanced other attributes.

https://geneformer.readthedocs.io/en/latest/geneformer.classifier.html

**Geneformer classifier.**

**Cell state classifier:**

Single-cell transcriptomes as Geneformer rank value encodings with cell state labels in Geneformer .dataset format (generated from single-cell RNAseq data by tokenizer.py)

**Gene classifier:**

Dictionary in format {Gene_label: list(genes)} for gene labels and single-cell transcriptomes as Geneformer rank value encodings in Geneformer .dataset format (generated from single-cell RNAseq data by tokenizer.py)

**Geneformer classifier.**


Parameters:


**classifier{“cell”, “gene”}**

Whether to fine-tune a cell state or gene classifier.

**quantize : bool, dict**

**cell_state_dict: None, dict**

Cell states to fine-tune model to distinguish.

Two-item dictionary with keys: state_key and states

**state_key:** key specifying name of column in .dataset that defines the states to model

**states:** list of values in the state_key column that specifies the states to model

Alternatively, instead of a list of states, can specify “all” to use all states in that state key from input data.

Of note, if using “all”, states will be defined after data is filtered.

Must have at least 2 states to model.

For example:

{“state_key”: “disease”,
“states”: [“nf”, “hcm”, “dcm”]}

or

{“state_key”: “disease”,
“states”: “all”}

**gene_class_dict:** None, dict

Gene classes to fine-tune model to distinguish.

Dictionary in format:

{Gene_label_A: list(geneA1, geneA2, …),

Gene_label_B: list(geneB1, geneB2, …)}

Gene values should be Ensembl IDs.

**filter_data** : None, dict

Default is to fine-tune with all input data.
Otherwise, dictionary specifying .dataset column name and list of values to filter by.

**rare_threshold** : float

Threshold below which rare cell states should be removed.
For example, setting to 0.05 will remove cell states representing
< 5% of the total cells from the cell state classifier’s possible classes.

**max_ncells** : None, int

Maximum number of cells to use for fine-tuning.

Default is to fine-tune with all input data.

**max_ncells_per_class** : None, int

Maximum number of cells per cell class to use for fine-tuning.
Of note, will be applied after max_ncells above.
(Only valid for cell classification.)

**training_args: None, dict

Training arguments for fine-tuning.
If None, defaults will be inferred for 6 layer Geneformer.

Note: Hyperparameter tuning is highly recommended, rather than using defaults.

**freeze_layers** : int

Number of layers to freeze from fine-tuning.

0: no layers will be frozen;

2: first two layers will be frozen; etc.

num_crossval_splits{0, 1, 5}

0: train on all data without splitting

1: split data into train and eval sets by designated split_sizes[“valid”]

5: split data into 5 folds of train and eval sets by designated split_sizes[“valid”]

**split_sizes** : None, dict

Dictionary of proportion of data to hold out for train, validation, and test sets
{“train”: 0.8, “valid”: 0.1, “test”: 0.1} if intending 80/10/10 train/valid/test split

**stratify_splits_col** : None, str

Name of column in .dataset to be used for stratified splitting.
Proportion of each class in this column will be the same in the splits as in the original dataset.

**no_eval** : bool
If True, will skip eval step and use all data for training.
Otherwise, will perform eval during training.

**forward_batch_sizeint**
Batch size for forward pass (for evaluation, not training).

**model_version : str

To auto-select settings for model version other than current default.

Current options:

V1: models pretrained on ~30M cells,

V2: models pretrained on ~104M cells

**token_dictionary_file** : None, str

Default is to use token dictionary file from Geneformer
Otherwise, will load custom gene token dictionary.

**🔧 Hyperparameters :**

**max_input_size = 2048**

Maximum number of tokens (genes) per cell input.

If a cell expresses more than 2048 genes, it will be truncated; if fewer, it may be padded.

**geneformer_batch_size = 5**

Number of examples (cells) processed in one forward/backward pass.

A small batch size reduces memory usage but may make training slower or noisier.

**max_lr = 5e-5**

Maximum learning rate (i.e., 0.00005).

Controls the step size during optimization. Too high can overshoot; too low may converge slowly.

**lr_schedule_fn = "linear"**

Learning rate schedule. "linear" means the learning rate will decay linearly from max_lr to 0 across training steps.

Helps the model stabilize after initial updates.

**warmup_steps = 500**

Number of steps over which the learning rate will increase linearly from 0 to max_lr.

Helps prevent sudden jumps at the beginning of training.

**epochs = 30**

Number of full passes through the training dataset.

More epochs mean more training time and potential overfitting if too high.

**optimizer = "adamw"**

Optimization algorithm. AdamW is a version of Adam that decouples weight decay from the gradient update.

Well-suited for transformer-based models.

**freeze_layers = 0**

Number of transformer layers to "freeze" (not update during training).

If set > 0, the first freeze_layers will be frozen; 0 means the full model is trainable

**🔍 What is TrainingArguments?**

TrainingArguments is a class provided by the Hugging Face transformers library. It defines all the parameters needed to train or evaluate a model using the Trainer API.

```
training_args = {
    "learning_rate": 5e-5,
    "do_train": True,
    "do_eval": True,
    "eval_strategy": "epoch",
    "save_strategy": "epoch",
    "logging_steps": 100,
    "group_by_length": True,
    "length_column_name": "length",
    "disable_tqdm": False,
    "lr_scheduler_type": "linear",
    "warmup_steps": 500,
    "weight_decay": 0.01,
    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size": 8,
    "num_train_epochs": 3,
    "load_best_model_at_end": True,

```


```
validate(model_directory, prepared_input_data_file, id_class_dict_file, output_directory, output_prefix, split_id_dict=None, attr_to_split=None, attr_to_balance=None, gene_balance=False, max_trials=100, pval_threshold=0.1, save_eval_output=True, predict_eval=True, predict_trainer=False, n_hyperopt_trials=0, save_gene_split_datasets=True, debug_gene_split_datasets=False)
```

**(Cross-)validate cell state or gene classifier.**

**Parameters**

**model_directoryPath**

Path to directory containing model

**prepared_input_data_filePath**

Path to directory containing _labeled.dataset previously prepared by Classifier.prepare_data

**id_class_dict_filePath**

Path to _id_class_dict.pkl previously prepared by Classifier.prepare_data
*(dictionary of format: numerical IDs: class_labels)*

**output_directoryPath**

Path to directory where model and eval data will be saved

**output_prefix**

Prefix for output files

**split_id_dict:** None, dict

Dictionary of IDs for train and eval splits

Three-item dictionary with keys: attr_key, train, eval

**attr_key:**

key specifying name of column in .dataset that contains the IDs for the data splits

**train:** list of IDs in the attr_key column to include in the train split

**eval:** list of IDs in the attr_key column to include in the eval split

For example:

{“attr_key”: “individual”,

“train”: [“patient1”, “patient2”, “patient3”, “patient4”],

“eval”: [“patient5”, “patient6”]}

Note: only available for CellClassifiers with 1-fold split (self.classifier=”cell”; self.num_crossval_splits=1)

**attr_to_split** : None, str

Key for attribute on which to split data while balancing potential confounders
e.g. “patient_id” for splitting by patient while balancing other characteristics

Note: only available for CellClassifiers with 1-fold split (self.classifier=”
cell”; self.num_crossval_splits=1)

**attr_to_balance** : None, list

List of attribute keys on which to balance data while splitting on attr_to_split
e.g. [“age”, “sex”] for balancing these characteristics while splitting by patient

**gene_balance** : None, bool

Whether to automatically balance genes in training set.
Only available for binary gene classifications.

**max_trials** : None, int

Maximum number of trials of random splitting to try to achieve balanced other attribute

If no split is found without significant (p < pval_threshold) differences in other attributes, will select best

**pval_threshold** : None, float

**save_eval_output** : bool

Whether to save cross-fold eval output
Saves as pickle file of dictionary of eval metrics

**predict_eval**: bool

Whether or not to save eval predictions

Saves as a pickle file of self.evaluate predictions

**predict_trainer**: bool

Whether or not to save eval predictions from trainer
Saves as a pickle file of trainer predictions

**n_hyperopt_trials** : int

Number of trials to run for hyperparameter optimization

If 0, will not optimize hyperparameters

**save_gene_split_dataset** : bool

Whether or not to save train, valid, and test gene-labeled datasets

In [ ]:
import pandas as pd

# Save the training set
# pancreas_trainset

# Save the evaluation set
# pancreas_evalset

# Load dataset from disk to verify
# pancreas_test_dataset = load_from_disk("/content/pancreas_shuffled_filtered_labeled_downsampled2000.train.dataset")
# print(pancreas_test_dataset)
# pancreas_train_dataset = load_from_disk("/content/pancreas_shuffled_filtered_labeled_downsampled2000.eval.dataset")
# print(pancreas_train_dataset)

# Print basic structure
# print(pancreas_train_dataset)
# print(pancreas_train_dataset.column_names)
# Inspect the first example
# print(pancreas_train_dataset[0])
# See metadata keys for one row
# example = pancreas_train_dataset[0]
# print(example.keys())

# If available:
# print(example.get("celltype"))
# print(example.get("label"))

# import pandas as pd

# df = pd.DataFrame(pancreas_train_dataset[:10])  # preview first 10 examples
# print(df.columns)
# print(df.head())

# Print basic structure
# print(pancreas_test_dataset)
# print(pancreas_test_dataset.column_names)

# Inspect the first example
# print(pancreas_test_dataset[0])

# See metadata keys for one row
# example_test = pancreas_test_dataset[0]
# print(example_test.keys())

# If available:
# print(example_test.get("celltype"))
# print(example_test.get("label"))

# Optional: Convert to DataFrame for inspection


# df_test = pd.DataFrame(pancreas_test_dataset[:10])  # preview first 10 examples
# print(df_test.columns)
# print(df_test.head())

# Optional: Convert to DataFrame for inspection
# df_test = pd.DataFrame(pancreas_test_dataset[:10])  # preview first 10 examples
# print(df_test.columns)
# print(df_test.head())

To keep or remove the field **"label"** from the metadata ?

Your dataset currently has a column named "label", which is reserved internally by Geneformer for classification targets (i.e., class IDs it creates itself during processing). If your dataset includes a column named "label" (like a duplicate of celltype), it conflicts with Geneformer's processing logic.

You need to rename or remove the "label" column from your dataset before calling cc.prepare_data().

In [ ]:
# The dataset with removed "label

# Remove the conflicting 'label' column
# if "label" in dataset.column_names:
#    dataset = dataset.remove_columns("label")

# Save updated dataset
# dataset.save_to_disk("/content/pancreas_shuffled_filtered_labeled_downsampled_fixed2.dataset")

# Confirm the "label" is gone
# print("After:", dataset.column_names)

In [ ]:
import torch
torch.cuda.empty_cache()

To add the label **"split"** in the metadata ?

In [ ]:
# Assign proper "split" values (optional but recommended)
# You can assign "train", "eval", and "test" splits like this:

# from datasets import Dataset
# import random

# Load your dataset
# dataset_path = "/content/pancreas_shuffled_filtered_labeled_downsampled.dataset" # Adjust path if needed fixed_dataset_path = "/content/pancreas_shuffled_filtered_labeled_downsampled_fixed.dataset"
# dataset = load_from_disk(dataset_path)

# Set seed for reproducibility
# random.seed(42)

# Shuffle and split
# dataset = dataset.shuffle(seed=42)
# n_total = len(dataset)
# n_train = int(n_total * 0.7)
# n_eval = int(n_total * 0.15)
# n_test = n_total - n_train - n_eval

# Assign split values
# split_labels = ["train"] * n_train + ["eval"] * n_eval + ["test"] * n_test
# dataset = dataset.map(lambda example, idx: {"split": split_labels[idx]}, with_indices=True)